In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import *

In [0]:
lista_moedas = ['EUR','DOL']

In [0]:
# Variáveis
data_inicial = '01-01-2022'
# data_final = '12-31-2021'
data_final = datetime.now().strftime('%m-%d-%Y') 
top = 100
skip = 0 

for moeda in lista_moedas:
    skip = 0
    todos_dados = []
    while True:

        url = (
            f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
            f"CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
            f"@moeda='{moeda}'&@dataInicial='{data_inicial}'&@dataFinalCotacao='{data_final}'"
            f"&$top={top}&$skip={skip}&format=json&$select=cotacaoCompra,dataHoraCotacao"
        )
        response = requests.get(url)
        dados = response.json()['value']
        if not dados:
            break
        todos_dados.extend(dados)
        skip += top

    if todos_dados:
        df = spark.createDataFrame(todos_dados) \
            .withColumn('moeda', lit(moeda))

    data_inicial_path = datetime.strptime(data_inicial, "%m-%d-%Y").strftime("%Y%m%d")
    data_final_path = datetime.strptime(data_final, "%m-%d-%Y").strftime("%Y%m%d")

    path = (
        f"Files/Cotacoes/Novos/"
        f"{moeda}-"
        f"{data_inicial_path}_"
        f"{data_final_path}"
        f".parquet"
    )

    #df.write.mode('overwrite').parquet(path)
display(df)